# LTX 2.3 I2V — Eros (MrXin) V6 — ComfyUI interaktif deneme (Colab)

**Input:** Başlangıç görseli + prompt (ComfyUI UI'da)
**Output:** 24 FPS video + ses (dual-pass)

Sıra:
1. **CONFIG** — Civitai cookie
2. **Helpers** — indirme + doğrulama yardımcıları
3. **ComfyUI + Manager + custom node'lar**
4. **Modeller** — önce gated (probe + indir), sonra HF
5. **Başlat + cloudflared tünel** → UI linki

> Drive kullanılmaz; modeller her oturumda kaynaktan iner (Colab ephemeral disk).
> **Runtime → Run all** → en alttaki linke gir → ComfyUI'da bu klasördeki `workflow.json`'u yükle → modelleri dropdown'dan seç → Run.

In [ ]:
# === CONFIG ===
# Civitai login-gated indirme: civitai.red → giriş → F12 → Application →
# Cookies → __Secure-civitai-token değerini yapıştır (uzun JWT benzeri string).
# SADECE bu cookie; ?token= API key KULLANMA (gated asset 401 verir).
COOKIE_VALUE = ""  # "__Secure-civitai-token" değeri

USE_GOOGLE_DRIVE = False  # Drive kapalı — modeller /content'e iner
COMFY_PORT = 8188

import os
os.environ["COOKIE_VALUE"] = COOKIE_VALUE
assert len(COOKIE_VALUE) > 500, "❌ COOKIE_VALUE boş/çok kısa — civitai.red'den __Secure-civitai-token yapıştır"
print(f"✓ Cookie: {len(COOKIE_VALUE)} char")
print("=== GPU ===")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
# === Shared helpers — log + fail-loud run + safetensors validation + civitai access ===
import os, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024: return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def banner(msg):
    """Section banner — makes pasted Colab output easy to read/debug."""
    print(f"\n{'='*60}\n# {msg}\n{'='*60}")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit/timeout -> RuntimeError (fail-loud)."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def is_valid_safetensors(path):
    """Real model file? -> (ok, msg). Empty/HTML/JSON login pages are caught as corrupt."""
    if not os.path.exists(path): return False, "yok"
    sz = os.path.getsize(path)
    if sz < 1_000_000: return False, f"çok küçük ({human(sz)})"
    with open(path, "rb") as f: head = f.read(8)
    if head.startswith(b"<") or head.startswith(b'{"'): return False, "HTML/JSON hata sayfası"
    try:
        jl = struct.unpack("<Q", head)[0]
        if 100 < jl < 200_000_000: return True, f"valid ({human(sz)})"
    except Exception: pass
    return False, "header bozuk"

# Civitai auth: session cookie ONLY (logged-in user). ?token= API key -> gated 401.
def civitai_url(version_id):
    return f"https://civitai.com/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civitai-token={COOKIE_VALUE}"

def civitai_probe(version_id, label):
    """Fail-fast: range-download first 1KB of a gated asset to verify access BEFORE heavy downloads.
    Missing/expired cookie -> login wall (HTML or non-2xx) -> RuntimeError immediately."""
    out = "/content/_probe.bin"
    code = (run(["curl", "-sL", "--max-time", "60", "-r", "0-1023",
                 "-H", cookie_header(), "-w", "%{http_code}", "-o", out, civitai_url(version_id)],
                f"probe {label}") or "").strip()[-3:]
    head = b""
    if os.path.exists(out):
        with open(out, "rb") as f: head = f.read(8)
        os.remove(out)
    if not code.startswith("2") or head.startswith(b"<") or head.startswith(b'{"'):
        raise RuntimeError(f"❌ Civitai erişimi başarısız: {label} (HTTP {code}). "
                           f"Cookie eksik/expired → civitai.red'den __Secure-civitai-token yenile, CONFIG'i tekrar çalıştır.")
    log(f"{label}: erişim OK", "OK")

def fetch(url, target_dir, filename, label, *, parallel, headers=None):
    """Download + validate; skip if already valid; raise on invalid (fail-loud).
    parallel=True -> aria2c (HF), False -> curl (Civitai, surfaces HTTP status)."""
    os.makedirs(target_dir, exist_ok=True)
    target = os.path.join(target_dir, filename)
    ok, msg = is_valid_safetensors(target)
    if ok: log(f"{label}: zaten var ({msg})"); return
    if os.path.exists(target): os.remove(target)
    log(f"{label}: indiriliyor...")
    t0 = time.time()
    if parallel:
        cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--console-log-level=warn",
               "--auto-file-renaming=false", "--allow-overwrite=true", "-d", target_dir, "-o", filename]
        if headers: cmd += ["--header", headers]
        cmd.append(url); run(cmd, label)
    else:
        cmd = ["curl", "-sL", "--max-time", "1800", "-w", "%{http_code}", "-o", target]
        if headers: cmd += ["-H", headers]
        cmd.append(url)
        code = (run(cmd, label) or "").strip()[-3:]
        if not code.startswith("2"):
            snippet = ""
            if os.path.exists(target):
                with open(target, "rb") as f: snippet = f.read(200).decode("utf-8", "replace").replace("\n", " ").strip()
            raise RuntimeError(f"{label}: Civitai HTTP {code} — {snippet}")
    ok, msg = is_valid_safetensors(target)
    if not ok: raise RuntimeError(f"{label}: indirme bozuk ({msg}) — {url.split('?')[0]}")
    log(f"{label}: indirildi ({msg}, {time.time()-t0:.0f}s)", "OK")

print("✓ Helpers hazır (banner, log, run, is_valid_safetensors, civitai_probe, fetch)")

In [ ]:
banner("3) ComfyUI + Manager + custom node'lar")
%cd /content
!apt-get install -y ffmpeg aria2 > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!echo "ComfyUI commit:" $(git -C /content/ComfyUI rev-parse --short HEAD)

import os
cn = "/content/ComfyUI/custom_nodes"
os.makedirs(cn, exist_ok=True)

# (folder, repo) — workflow.json'dan çıkarılan 16 paket; URL'ler ComfyUI registry/GitHub ile doğrulandı.
# ComfyUI-Manager ilk sırada (eksik node'ları UI'da yakalamak için).
NODE_REPOS = [
    ("ComfyUI-Manager",            "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-KJNodes",            "https://github.com/kijai/ComfyUI-KJNodes.git"),
    ("rgthree-comfy",              "https://github.com/rgthree/rgthree-comfy.git"),
    ("ComfyUI-Easy-Use",           "https://github.com/yolain/ComfyUI-Easy-Use.git"),
    ("ComfyUI-mxToolkit",          "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),
    ("ComfyUI-VideoHelperSuite",   "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),
    ("ComfyUI-LTXVideo",           "https://github.com/Lightricks/ComfyUI-LTXVideo.git"),
    ("ComfyUI-Impact-Pack",        "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),
    ("ComfyUI-VFI",                "https://github.com/GACLove/ComfyUI-VFI.git"),
    ("RES4LYF",                    "https://github.com/ClownsharkBatwing/RES4LYF.git"),
    ("ComfyUI-Custom-Scripts",     "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),
    ("ComfyLiterals",              "https://github.com/M1kep/ComfyLiterals.git"),
    ("Comfyui-Memory_Cleanup",     "https://github.com/LAOGOU-666/Comfyui-Memory_Cleanup.git"),
    ("ControlAltAI-Nodes",         "https://github.com/gseth/ControlAltAI-Nodes.git"),
    ("comfyui-int-and-float",      "https://github.com/danTheMonk/comfyui-int-and-float.git"),
    ("ComfyUI_NVIDIA_RTX_Nodes",   "https://github.com/Comfy-Org/Nvidia_RTX_Nodes_ComfyUI.git"),
]

# Tolerant: bir URL ölü/yanlışsa notebook durmasın — WARN bas, devam et (Manager UI'da yakalar).
fail = []
for name, url in NODE_REPOS:
    dest = os.path.join(cn, name)
    if os.path.isdir(dest) and os.listdir(dest):
        log(f"{name}: zaten var"); continue
    log(f"{name}: cloning...")
    try:
        run(["git", "clone", "--depth", "1", url, dest], f"clone {name}", timeout=180)
        req = os.path.join(dest, "requirements.txt")
        if os.path.isfile(req):
            run(["pip", "install", "-q", "-r", req], f"pip {name}", timeout=300)
        log(f"{name}: OK", "OK")
    except RuntimeError as e:
        fail.append(name)
        log(f"{name}: BAŞARISIZ → UI'da Manager ile kur. ({str(e).splitlines()[-1][:120]})", "WARN")
log(f"{len(NODE_REPOS)-len(fail)}/{len(NODE_REPOS)} node hazır" + (f" | başarısız: {fail}" if fail else ""), "OK")

In [ ]:
banner("4) Modeller — önce gated (probe + indir), sonra HF")
import glob
COMFY = "/content/ComfyUI/models"

# (version_id, subfolder, filename, label) — gated, cookie ile iner
CIVITAI_MODELS = [
    (2892069, "diffusion_models", "ltx2310eros_v1_FP8.safetensors", "LTX Eros checkpoint"),
    (164677,  "upscale_models",   "nmkdSiaxCX_200k.safetensors",    "nmkdSiaxCX upscaler"),
    # Concept LoRA'lar (workflow Power Lora Loader'larında; Civitai version ID + dosya adı doğrulandı)
    (2950842, "loras", "DR34ML4Y_LTXXX_V2.safetensors", "DR34ML4Y LoRA"),
    (2772932, "loras", "Penile_Praxis_V4.safetensors",  "Penile Praxis LoRA"),
    (2996907, "loras", "DaSiWa_LTX23_NSFW_Bodyphysics_Fluid_Motion_Enhancer_v01.safetensors", "Body Physics LoRA"),
]

# (url, subfolder, filename, label) — public HF/GitHub direct link (workflow "Model Links" node'undan)
HF_MODELS = [
    ("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/diffusion_models/ltx-2.3-22b-distilled_transformer_only_fp8_input_scaled_v3.safetensors", "diffusion_models", "ltx-2.3-22b-distilled_transformer_only_fp8_input_scaled_v3.safetensors", "LTX distilled"),
    ("https://huggingface.co/GitMylo/LTX-2-comfy_gemma_fp8_e4m3fn/resolve/main/gemma_3_12B_it_fp8_e4m3fn.safetensors", "text_encoders", "gemma_3_12B_it_fp8_e4m3fn.safetensors", "Gemma text encoder"),
    ("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/text_encoders/ltx-2.3_text_projection_bf16.safetensors", "clip", "ltx-2.3_text_projection_bf16.safetensors", "Text projection"),
    ("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/LTX23_video_vae_bf16.safetensors", "vae", "LTX23_video_vae_bf16.safetensors", "Video VAE"),
    ("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/LTX23_audio_vae_bf16.safetensors", "vae", "LTX23_audio_vae_bf16.safetensors", "Audio VAE"),
    ("https://huggingface.co/madebyollin/taehv/resolve/main/safetensors/taeltx2_3.safetensors", "vae", "taeltx2_3.safetensors", "Preview VAE"),
    ("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.1.safetensors", "Spatial upscaler"),
    ("https://huggingface.co/TenStrip/LTX2.3_Distilled_Lora_1.1_Experiments/resolve/main/ltx-2.3-22b-distilled-lora-1.1_fro90_ceil72_condsafe.safetensors", "loras", "ltx-2.3-22b-distilled-lora-1.1_fro90_ceil72_condsafe.safetensors", "Distilled LoRA first"),
    ("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384-1.1.safetensors", "loras", "ltx-2.3-22b-distilled-lora-384-1.1.safetensors", "Distilled LoRA final"),
]

# 1) Fail-fast: gated erişimini önce doğrula (hiçbir büyük dosya inmeden)
log(f"Gated probe: {len(CIVITAI_MODELS)} asset, HF: {len(HF_MODELS)} dosya")
for vid, sub, fn, label in CIVITAI_MODELS:
    civitai_probe(vid, label)

# 2) Gated modelleri önce indir (inmeme ihtimali en yüksek olanlar)
for vid, sub, fn, label in CIVITAI_MODELS:
    fetch(civitai_url(vid), os.path.join(COMFY, sub), fn, label, parallel=False, headers=cookie_header())

# 3) HF modelleri (aria2c)
for url, sub, fn, label in HF_MODELS:
    fetch(url, os.path.join(COMFY, sub), fn, label, parallel=True)

# === Özet — buraya ulaşmak = her şey indi + doğrulandı ===
banner("İndirme özeti")
for f in sorted(glob.glob(f"{COMFY}/**/*", recursive=True)):
    if os.path.isfile(f):
        print(f"   {human(os.path.getsize(f)):>9}  {os.path.relpath(f, COMFY)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

In [ ]:
import subprocess, time, urllib.request, threading, re, os
banner("5) ComfyUI başlat + cloudflared tünel")

if not os.path.isfile("/content/cloudflared"):
    run(["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], "cloudflared")
    run(["chmod", "+x", "/content/cloudflared"], "chmod cloudflared")

subprocess.run(["pkill", "-f", "main.py"], check=False); time.sleep(2)
logf = open("/content/comfyui.log", "w")
subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                 cwd="/content/ComfyUI", stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=2); ok = True; break
    except Exception: pass
if not ok:
    print("".join(open("/content/comfyui.log").readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı")
log(f"ComfyUI ayakta ({(i+1)*2}s)", "OK")

tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{COMFY_PORT}"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
def _url():
    for line in tun.stdout:
        m = re.search(r"https://[-\w.]+trycloudflare\.com", line)
        if m: print("\n🔗 ComfyUI linki:", m.group(0), "\n"); break
threading.Thread(target=_url, daemon=True).start(); time.sleep(8)
print("⬆️ Link yukarıda. Linke gir, workflow.json'u yükle, modelleri seç, Run.")

## Kullanım

1. Yukarıdaki `trycloudflare.com` linkine gir (ComfyUI UI açılır).
2. **Workflow → Open** ile bu klasördeki `workflow.json`'u yükle (veya dosyayı tarayıcıya sürükle).
3. Her model loader'a tıklayıp dropdown'dan indirilen dosyayı seç (yollar otomatik çözülmez). LTX dosyaları düz `diffusion_models/`, `vae/`, `loras/` vb. altına iner.
4. **Run**. Oturum koparsa **Run all** tekrar — inen modeller atlanır.

### Notlar
- **Checkpoint seçimi:** workflow "Switch Model" ile 10Eros FP8 ↔ Distilled 22B arası geçiş yapar. İlk denemede 10Eros (`ltx2310eros_v1_FP8.safetensors`) önerilir.
- **Final pass LoRA:** `ltx-2.3-22b-distilled-lora-384-1.1.safetensors` (workflow notu).
- **Concept LoRA'lar (Power Lora Loader):** `DR34ML4Y_LTXXX_V2` ve `Penile_Praxis_V4` indiriliyor (isim birebir). **Physics** için indirilen dosya `DaSiWa_LTX23_NSFW_Bodyphysics_Fluid_Motion_Enhancer_v01.safetensors` — workflow'daki placeholder adı (`LTX2.3_Physics_V2_...`) farklı; Physics loader'ında dropdown'dan bunu seç.
- **OOM olursa:** Preview'i kapat, Chunk'ı kapat (workflow içi "Tip" node'ları). Colab A100'de (40 GB) genelde gerekmez.
- Eksik custom node olursa: UI'da **Manager → Install Missing Custom Nodes** → Restart. (Notebook node clone'da hata olursa durmaz, WARN basıp devam eder.)